# 12 インタラクティブ・ダッシュボード(Plotly)

`irp.visualization` のフィギュアビルダーは、プラットフォームが既に作る オブジェクト(`BacktestResult`・`compare` 表・IC・コストスイープ)から **統一テーマ**の Plotly 図を返す。ノートブックでは `fig.show()` で対話的に、レポートでは HTML に埋め込む。

In [ ]:
import numpy as np
import pandas as pd
from irp import signals as S, backtest as B, visualization as V
from irp.features.price import cross_sectional_zscore

# Synthetic panel with a mild persistent edge (no network). Real data:
# irp.data.price_panel. Figures below work the same on real backtests.
rng = np.random.default_rng(11)
idx = pd.bdate_range('2015-01-01', periods=1500)
names = [f'A{i}' for i in range(10)]
edge = np.linspace(-0.0003, 0.0004, len(names))
rets = pd.DataFrame(
    {n: rng.normal(edge[i], 0.012, len(idx)) for i, n in enumerate(names)}, index=idx
)
prices = 100 * np.exp(rets.cumsum())
# momentum long-short, monthly rebalanced; vs equal-weight benchmark
sig = S.momentum_signal(prices, lookback=252, skip=21)
w = B.rebalanced(S.long_short_quantile(sig.lag(1).score, 0.3), 'ME')
strategy = B.run_backtest(w, rets, cost_model=B.CostModel(5, 2), lag=1)
equal_weight = B.buy_and_hold(rets)
results = {'momentum_LS': strategy, 'equal_weight': equal_weight}
print('built:', list(results), '| OOS days:', len(strategy.returns))

## エクイティ曲線 & ドローダウン

In [ ]:
V.equity_curves(results).show()

In [ ]:
V.drawdown(strategy).show()

## 指標表(戦略 × ベースライン)

In [ ]:
V.metrics_table(B.compare(results)).show()

## リターン分布 & ローリング Sharpe

In [ ]:
V.returns_histogram(strategy).show()

In [ ]:
V.rolling_sharpe(strategy, window=126).show()

## コスト感応度(ターンオーバーの効き)

In [ ]:
rows = []
for bps in [0, 5, 10, 20, 50]:
    r = B.run_backtest(w, rets, cost_model=B.CostModel(bps, 0), lag=1)
    rows.append({'cost_bps': bps, 'sharpe': B.sharpe(r, 252)})
sweep = pd.DataFrame(rows).set_index('cost_bps')
V.cost_sensitivity(sweep).show()

同じ図群を `V.strategy_report(...)` でオフライン HTML に束ねる(NB13)。合成データなので数値は説明用で、結果そのものではない。